# Lec12: Machine Learning with Scikit-learn II

### Multiple Linear Regression

In [ ]:
import matplotlib.pyplot as plt
import seaborn; seaborn.set() # plot formatting
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.RandomState(1)
X = 10 * rng.rand(100, 3)     # [X1, X2, X3]; shape (100,3)
y = 0.5 + np.dot(X, [1.5, -2., 1.])  # y= 0.5 + 1.5*X1 - 2*X2 + X3

model = LinearRegression()
model.fit(X, y)
print(model.intercept_)
print(model.coef_)



Here the $y$ data is constructed from three random $x$ values, and the linear regression recovers the coefficients used to construct the data.
$$
y = 0.5 + 1.5x_1 - 2x_2 + x_3
$$

The ``LinearRegression`` estimator, in addition to simple straight-line fits, it can also handle multidimensional linear models of the form
$$
y = a_0 + a_1 x_1 + a_2 x_2 + \cdots
$$
where there are multiple $x_n$ values(features).
In this way, we can use the single ``LinearRegression`` estimator to fit lines, planes, or hyperplanes to our data. <br>
It still appears that this approach would be limited to strictly linear relationships between variables, 
but it turns out we can extend it to polynomial function regression.


### Polynomial regression

In [ ]:
# create a dataset with y depends on x in form of 2nd degree polynomial

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline  

def make_data(N, err=1.0, rseed=1):  
    # randomly sample the data
    rng = np.random.RandomState(rseed)
    X = rng.rand(N, 1) ** 2
    y = 10 - 1. / (X.ravel() + 0.1)
    if err > 0:
        y += err * rng.randn(N)
    return X, y
X, y = make_data(40)  # X's shape is (40,1)
X_1D = X.ravel()  #flatten a multi-dimensional array into a 1D array; X_1D's shape is (40,)
plt.scatter(X_1D, y, color='black')


The above data points need a polynomial of degree n to do the regression<br>
$$
y = a_0 + a_1 x + a_2 x² + a_3 x³\cdots
$$
if we transform the polynomial equation to a multidimensional linear equation 
$$
y = a_0 + a_1 x_1 + a_2 x_2 +  a_3x_3\cdots
$$
$$
i.e.   x=x_1,  x²=x_2,  x³=x_3 \cdots
$$

Assume we want to use a 3rd degree polynomial to do the regression, then the orignal feature array
X where X's shape is (40,1) now need to be transformed to a new array X_new where X's shape is (40,3) with three columns [x, x², x³]. <br>
We can use the PolynomialFeatures transformer to do the transformation. For example:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
x_old = np.array([2, 3, 4])
poly_3 = PolynomialFeatures(3, include_bias=False)
X_new = poly_3.fit_transform(x_old[:, None])  #transform a 1-D array to a 2-D array with degree=3
print (X_new)

The PolynomialFeatures object poly_3 can generate new features from existing feature by creating polynomial  terms.
This new, higher-dimensional data representation X_new can then be plugged into a linear regression model and used in training.

If we have a feature array with multiple features (columns), then the transformed feature array will contain combined higher-order features. For example:  <br>  
Assume we have a dataset X which has two features [a,b] and a poly_2 object formed by: <br> 
    poly_2 = PolynomialFeatures(2, include_bias=False) <br>
By calling the object method fit_transform(X) <br> 
    new_X = poly_2.fit_transform(X)  <br>
we can transform data X to new_X with higher degree features [1, a, b, a², ab, b²]<br>
    
the new_X now has 6 columns [ 1, a, b, a², ab, b²] ;the first column has all 1 to account of the intercept <br>
then  new_X is used to train a  a linear regression model and  a multi-dimensional linear function<br>
$$
y = m_0 + m_1x_1 + m_2x_2 + m_3x_3 + m_4x_4 + m_5x_5 
$$  
can be created where [x1, x2, x3, x4, x5] = [ a, b, a², ab, b²] <br>
which is equvilant to a  polynomial function:  
    y = m0 + m1·a + m2·b + m3·a² + m4·ab + + m5·b² <br>
Note: if degree=3; original features [a,b] => new features [1, a, b, a², ab, b², a³, a²b, ab², b³]

####  However, the cleanest way to accomplish these is to use a Pipeline. 

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
# create a pipeline of methods for sequential execution 
def PolynomialRegression(degree=2, **kwargs):
    return make_pipeline(PolynomialFeatures(degree),
                         LinearRegression(**kwargs))


the  **PolynomialRegression() function** return a pipeline object

pipeline =  make_pipeline(PolynomialFeatures(2),LinearRegression()) <br>

the make_pipeline function call the two constructor methods to create two objects <br>
a **PolynomialFeatures** object (call it 'poly_2') and a **LinearRegression** object (call it 'model_LR'):<br>

poly_2 = PolynomialFeatures(2)     #create a 2nd degree (by default) Polynomial Transformer object<br>
model_LR = LinearRegression()     # create a Linear Regression Model object<br>


In [ ]:

X_test = np.linspace(-0.1, 1.1, 500)[:, None]
plt.scatter(X.ravel(), y, color='black')
axis = plt.axis()
for degree in [1, 3, 5]:
    pipeline_object = PolynomialRegression(degree) 
    """
    the pipeline object execute the two constructor methods to create two objects
    a PolynomialFeatures object (call it 'poly') and a LinearRegression object (call it 'model_LR') 
    """
    pipeline_object.fit(X, y)
    """
    the pipeline_object.fit() method first call the poly.fit_transform(X) method which create a new X 
    with new features (call it 'new_X'); then it execute the model_LR.fit(new_X) method to train the model
        new_X = poly.fit_transform(X)  # transform the data X to new_X with high degree features
        model_LR.fit(new_X) # train the model base on the new dataset
        the model_LR will create a multi-dimensional linear function y where   
           y = m0 + m1·x1 + m2·x2 + + m3·x3  # if degree is 3 => [1,x1,x2,x3] = [1, x, x², x³]
    after training, the best fit parameter set (m0,m1,m2,m3) is found 
    now the model has a polynomial regression function y = m0 + m1·x + m2·x² + m3·x³ 
    """
    y_test = pipeline_object.predict(X_test)
    """
    the pipeline_object.predict(X_test) method first call the poly.fit_transform(X_test) which create a new X 
    with new features; then it call the model_LR.predict(new_X) method and return the predicted result y_test
        new_X = poly.fit_transform(X_test)  # transform the data X_test to new_X with high degree features
        y_test = model_LR.predict(new_X)    # predict the result using the multi-dimensional linear function    
                                            #  y = m0 + m1·new_X[0] + m2·new_X[1] + m3·new_X[2]
    """
    plt.plot(X_test.ravel(), y_test, label='degree={0}'.format(degree))
plt.xlim(-0.1, 1.0)
plt.ylim(-2, 12)
plt.legend(loc='best');
plt.show()

## Model selection: cross-validation and bias variance trade-off

In [ ]:
from sklearn.model_selection import validation_curve
degree = np.arange(0, 21)
train_score, val_score = validation_curve(PolynomialRegression(), X, y,
                                          param_name='polynomialfeatures__degree', param_range=degree, cv=7)

plt.plot(degree, np.median(train_score, 1), color='blue', label='training score')
plt.plot(degree, np.median(val_score, 1), color='red', label='validation score')
plt.legend(loc='best')
plt.ylim(0, 1)
plt.xlabel('degree')
plt.ylabel('score');

### The Bias–Variance Trade-off
Expected test error splits into three parts:
- error from a model that's too simple (bias), high bias causes a model to miss important relationships in the data, resulting in underfitting. 
- error from a model that's too sensitive to the particular training sample (variance), high variance causes a model to memorize random noise rather than the underlying pattern, resulting in overfitting.
- and noise you can never remove

Pushing one down usually pushes the other up.
In our case: <br>
  degree < 2  => underfitting <br>
  degree > 4  => overfitting <br>
  degree = 3     => optimum/balance 

In [ ]:
plt.scatter(X.ravel(), y)
lim = plt.axis()
y_test = PolynomialRegression(3).fit(X, y).predict(X_test)
plt.plot(X_test.ravel(), y_test);
plt.axis(lim);
plt.show()

In [ ]:
model = pipeline.steps[-1][1]  # Access the model by index (model is the last step)
print("Model slopes:    ", model.coef_[0],model.coef_[1],model.coef_[2],model.coef_[3],model.coef_[4],model.coef_[5])
print("Model intercept:", model.intercept_)

## Application: Exploring Handwritten Digits

#### Loading and visualizing the digits data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.datasets import load_digits
digits = load_digits()
digits.images.shape

In [ ]:
digits.target.shape

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(10, 10, figsize=(8, 8)
                         ,subplot_kw={'xticks':[], 'yticks':[]}
                         )
for i, ax in enumerate(axes.flat):    #axes.flat is a  1-D flat iterator (100,) of the subplots (10 x 10)
    ax.imshow(digits.images[i], cmap='binary', interpolation='nearest')  #display data as an image on a 2D regular raster
    ax.text(0.05, 0.05, str(digits.target[i]),
             transform=ax.transAxes,color='green')

plt.show()

#### Dimension reduction: manifold learning algorithm

In [ ]:
from sklearn.manifold import Isomap # Isometric Mapping is a non-linear dimensionality reduction
iso = Isomap(n_components=2)
iso.fit(digits.data)
data_projected = iso.transform(digits.data)
data_projected.shape

In [ ]:
plt.scatter(data_projected[:, 0], data_projected[:, 1], c=digits.target,
            cmap='Reds')
plt.colorbar(label='digit label', ticks=range(10));
plt.show()

#### Classification

In [ ]:
y.shape

In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, random_state=0)

In [ ]:

from sklearn.naive_bayes import GaussianNB
model = GaussianNB()
model.fit(Xtrain, ytrain)
y_model = model.predict(Xtest)

from sklearn.metrics import accuracy_score
accuracy_score(ytest, y_model)

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
mat = confusion_matrix(ytest, y_model)
sns.heatmap(mat, square=True, annot=True, cbar=False)
plt.xlabel('predicted value')
plt.ylabel('true value');
plt.show()

In [ ]:
fig, axes = plt.subplots(10, 10, figsize=(8, 8),
subplot_kw={'xticks':[], 'yticks':[]},
gridspec_kw=dict(hspace=0.1, wspace=0.1))
    
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='binary', interpolation='nearest')
    ax.text(0.05, 0.05, str(digits.target[i]),
            transform=ax.transAxes, color='green' if (ytest[i] == y_model[i]) else 'red')
plt.show()